In [18]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard component imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64

JupyterDash.infer_jupyter_proxy_config()

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from animal_shelter import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "password"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=['_id'], inplace=True, errors='ignore')

# Rescue type filter queries based on Grazioso Salvare breed specifications
water_rescue_query = {
    "animal_type": "Dog",
    "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
}

mountain_rescue_query = {
    "animal_type": "Dog",
    "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
}

disaster_rescue_query = {
    "animal_type": "Dog",
    "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
}


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([

    # Header: logo, title, and unique identifier
    html.Div([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '100px', 'float': 'left', 'marginRight': '20px'}
        ),
        html.Div([
            html.H1('Grazioso Salvare Animal Rescue Dashboard',
                    style={'textAlign': 'center'}),
            html.H4('Developed by Charles Drew Givens | CS-340',
                    style={'textAlign': 'center', 'color': 'gray'})
        ])
    ], style={'display': 'flex', 'alignItems': 'center', 'padding': '10px'}),

    html.Hr(),

    # Rescue type filter
    html.Div([
        html.B('Select Rescue Type:'),
        html.Br(),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue',                   'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue',  'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking','value': 'disaster'},
                {'label': 'Reset',                          'value': 'reset'},
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px'}
        )
    ], style={'padding': '10px'}),

    html.Hr(),

    # Interactive data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict('records'),
        page_action='native',
        page_current=0,
        page_size=10,
        sort_action='native',
        sort_mode='multi',
        filter_action='native',
        row_selectable='single',
        selected_rows=[],
        column_selectable='single',
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'padding': '5px',
            'minWidth': '80px',
            'maxWidth': '180px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis',
        },
        style_header={
            'backgroundColor': 'rgb(230, 230, 230)',
            'fontWeight': 'bold'
        },
        style_data_conditional=[
            {
                'if': {'row_index': 'odd'},
                'backgroundColor': 'rgb(248, 248, 248)'
            }
        ],
    ),

    html.Br(),
    html.Hr(),

    # Chart and geolocation map side by side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', className='col s12 m6'),
            html.Div(id='map-id',   className='col s12 m6'),
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    if filter_type == 'water':
        query = water_rescue_query
    elif filter_type == 'mountain':
        query = mountain_rescue_query
    elif filter_type == 'disaster':
        query = disaster_rescue_query
    else:
        query = {}

    df_filtered = pd.DataFrame.from_records(db.read(query))
    df_filtered.drop(columns=['_id'], inplace=True, errors='ignore')
    return df_filtered.to_dict('records')


@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    if viewData is None:
        return []
    dff = pd.DataFrame.from_dict(viewData)

    breed_counts = dff['breed'].value_counts()
    top_breeds = breed_counts.nlargest(10)
    other_count = breed_counts.iloc[10:].sum()

    if other_count > 0:
        top_breeds['Other'] = other_count

    chart_df = top_breeds.reset_index()
    chart_df.columns = ['breed', 'count']

    return [
        dcc.Graph(
            figure=px.pie(
                chart_df,
                names='breed',
                values='count',
                title='Breed Distribution'
            )
        )
    ]


@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    if viewData is None:
        return []
    elif index is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)
    row = 0 if not index else index[0]

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id='base-layer-id'),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(dff.iloc[row, 4]),
                        dl.Popup([
                            html.H1('Animal Name'),
                            html.P(dff.iloc[row, 9])
                        ])
                    ]
                )
            ]
        )
    ]


app.run_server()

Dash app running on https://parlorfamous-actordeluxe-3000.codio.io/proxy/8050/
